In [11]:
import numpy as np
import pandas as pd
df = pd.read_csv('customers_messy_500.csv')
print(df.head(5),df.tail(5),sep='\n')
print(df.shape,df.columns,df.dtypes,sep='\n')
print(df.info())
print(df.describe(include='all'))
#float type data as strings
#null data in every column
#multiple data for same customer

  customer_id   age     city    income  score  purchases purchased
0       C1176  51.0  Colombo   60128.0  0.728        2.0         0
1       C1132  22.0    Galle   94989.0  0.387        4.0         0
2       C1006   NaN   Jaffna   81217.0  0.751        0.0         0
3       C1084  47.0  COLOMBO  103321.0  0.575        6.0         1
4       C1456   NaN    Galle   58237.0  0.798        8.0         1
    customer_id   age     city   income  score  purchases purchased
513       C1071   NaN    Kandy  47493.0  0.541        6.0         0
514       C1106  54.0    Galle  75680.0   0.67        0.0         0
515       C1270  31.0   JAFFNA  94152.0  0.509        8.0         0
516       C1435  23.0  Colombo  77433.0  0.338       10.0         0
517       C1102  55.0    Kandy  49434.0  0.616       10.0        NO
(518, 7)
Index(['customer_id', 'age', 'city', 'income', 'score', 'purchases',
       'purchased'],
      dtype='str')
customer_id        str
age            float64
city               str
inc

In [12]:
print(df.isna().sum())
print(df.isna().sum()/df.shape[0]*100)
#higher percentage is in age column.So age is most incomplete.

customer_id     1
age            37
city           21
income         25
score          32
purchases       2
purchased       2
dtype: int64
customer_id    0.193050
age            7.142857
city           4.054054
income         4.826255
score          6.177606
purchases      0.386100
purchased      0.386100
dtype: float64


In [ ]:
print(df.loc[:9,['customer_id','age','city','score']])
print(df.iloc[:10,:4])

  customer_id   age       city  score
0       C1176  51.0    Colombo  0.728
1       C1132  22.0      Galle  0.387
2       C1006   NaN     Jaffna  0.751
3       C1084  47.0    COLOMBO  0.575
4       C1456   NaN      Galle  0.798
5       C1311  49.0    Colombo  0.676
6       C1333  57.0   Colombo   0.554
7       C1072  39.0    Colombo  0.549
8       C1184  41.0   Colombo   0.515
9       C1154  38.0    Negombo  0.616
  customer_id   age       city    income
0       C1176  51.0    Colombo   60128.0
1       C1132  22.0      Galle   94989.0
2       C1006   NaN     Jaffna   81217.0
3       C1084  47.0    COLOMBO  103321.0
4       C1456   NaN      Galle   58237.0
5       C1311  49.0    Colombo   79461.0
6       C1333  57.0   Colombo   128793.0
7       C1072  39.0    Colombo   86247.0
8       C1184  41.0   Colombo    61311.0
9       C1154  38.0    Negombo   33377.0


In [14]:
score_series = pd.Series(df.score)
age_score = pd.DataFrame({'age':df.age,'score':df.score})
print(score_series)
print(age_score)
#In loc we have to use labels as column indexing while iloc uses indexes as column indexing.

0      0.728
1      0.387
2      0.751
3      0.575
4      0.798
       ...  
513    0.541
514     0.67
515    0.509
516    0.338
517    0.616
Name: score, Length: 518, dtype: str
      age  score
0    51.0  0.728
1    22.0  0.387
2     NaN  0.751
3    47.0  0.575
4     NaN  0.798
..    ...    ...
513   NaN  0.541
514  54.0   0.67
515  31.0  0.509
516  23.0  0.338
517  55.0  0.616

[518 rows x 2 columns]


In [15]:
df.age = pd.to_numeric(df.age,errors='coerce')
df["score"] = df["score"].apply(
    lambda x: float(x.strip().replace("%", "")) / 100
    if isinstance(x, str) and "%" in x
    else pd.to_numeric(x, errors="coerce")
)
    

In [16]:
df["income"] = pd.to_numeric(df["income"].str.strip().str.replace("LKR", "").str.replace(',',''),errors="coerce")

In [17]:
print(df.dtypes)

customer_id        str
age            float64
city               str
income         float64
score          float64
purchases      float64
purchased          str
dtype: object


In [873]:
df.city = df.city.str.strip().str.capitalize().str.replace('Colmbo','Colombo').str.replace('Gale','Galle').replace('',np.nan)
print(df.city.value_counts())

city
Colombo    203
Kandy      104
Galle       93
Jaffna      61
Negombo     35
Name: count, dtype: int64


In [874]:
df.dropna(subset=['customer_id'],inplace=True)
df.dropna(thresh=4,inplace=True)

In [875]:
df.age = df.age.fillna(df.age.median())
df.score = df.score.fillna(df.score.median())
df.income = df.income.fillna(df.income.median())

In [876]:
df.city = df.city.fillna(df.city.mode()[0])
#If we drop NaN city rows we loose lot of important informations.
print(df.isna().sum())

customer_id    0
age            0
city           0
income         0
score          0
purchases      0
purchased      0
dtype: int64


In [877]:
before = len(df)
df = df[(df['age']>18) & (df['age']<80) & (df['score']<1) & (df['score']>0) & (df['purchases']>=0)]
print(f'No.of rows removed : {before-len(df)}')
print(df.shape)

No.of rows removed : 31
(484, 7)


In [878]:
df.purchased = df.purchased.apply(lambda x : 1 if x in ['Yes','YES','true','True','y','1']
                                  else 0 if x in ['No','NO','false','False','n','0']
                                  else pd.NA).dropna()

In [879]:
df.drop_duplicates(subset=['customer_id'],inplace=True,keep='first')
df.drop_duplicates(keep='first',inplace=True)

In [880]:
print(f'Final No.of rows : {df.shape[0]}')

Final No.of rows : 477


In [881]:
df['score_pct'] = df.score*100
df['high_value'] = df['income']>df.income.median()

In [882]:
df.sort_values('score', ascending=False, inplace=True, ignore_index=True)
print(df.head(10))

  customer_id  age     city    income  score  purchases  purchased  score_pct  \
0       C1226   56    Kandy   82389.0   0.98       11.0          1       98.0   
1       C1440   63  Colombo   75252.0   0.98        9.0          0       98.0   
2       C1068   21  Colombo   51187.0   0.98        2.0          1       98.0   
3       C1288   41  Colombo   91637.0   0.98        4.0          1       98.0   
4       C1236   49    Kandy   64351.0   0.98        6.0          1       98.0   
5       C1196   34  Negombo  106599.0   0.98        9.0          1       98.0   
6       C1125   44  Colombo   83040.0   0.98        0.0          1       98.0   
7       C1118   29  Colombo  115423.0   0.98       11.0          1       98.0   
8       C1396   20  Colombo   99941.0   0.98        5.0          1       98.0   
9       C1191   47  Colombo   83735.0   0.98        3.0          1       98.0   

   high_value  
0        True  
1       False  
2       False  
3        True  
4       False  
5        Tru

In [883]:
filtered_customers = df[(df.city=='Colombo') & (df.score>=0.7) & (df.purchased==1)]
print(filtered_customers)
print(len(filtered_customers))

    customer_id  age     city    income  score  purchases  purchased  \
2         C1068   21  Colombo   51187.0  0.980        2.0          1   
3         C1288   41  Colombo   91637.0  0.980        4.0          1   
6         C1125   44  Colombo   83040.0  0.980        0.0          1   
7         C1118   29  Colombo  115423.0  0.980       11.0          1   
8         C1396   20  Colombo   99941.0  0.980        5.0          1   
9         C1191   47  Colombo   83735.0  0.980        3.0          1   
11        C1134   55  Colombo   81129.0  0.964        6.0          1   
16        C1443   40  Colombo   41839.0  0.944        7.0          1   
19        C1259   41  Colombo   84764.0  0.927        6.0          1   
20        C1022   26  Colombo   52546.0  0.923        3.0          1   
24        C1019   39  Colombo   25560.0  0.899        7.0          1   
26        C1031   28  Colombo   68416.0  0.893        4.0          1   
28        C1171   41  Colombo   62730.0  0.886        4.0       

In [884]:
print(df.city.value_counts())

city
Colombo    205
Kandy       99
Galle       84
Jaffna      56
Negombo     33
Name: count, dtype: int64


In [885]:
print(df.purchased.value_counts())

purchased
0    329
1    148
Name: count, dtype: int64


In [886]:
print(df.value_counts(subset=['city','purchased']))

city     purchased
Colombo  0            140
Kandy    0             71
Colombo  1             65
Galle    0             56
Jaffna   0             42
Kandy    1             28
Galle    1             28
Negombo  0             20
Jaffna   1             14
Negombo  1             13
Name: count, dtype: int64


In [887]:
print(df.groupby('city')['score'].mean())

city
Colombo    0.612149
Galle      0.620845
Jaffna     0.591250
Kandy      0.582667
Negombo    0.586924
Name: score, dtype: float64


In [888]:
print(df.groupby('city')[['income','purchases']].mean())

               income  purchases
city                            
Colombo  75504.663415   5.429268
Galle    76415.666667   5.285714
Jaffna   73051.660714   5.553571
Kandy    71910.070707   5.393939
Negombo  75446.696970   5.969697


In [889]:
print(df.groupby('city').agg(
    row_count = ('customer_id','count'),
    mean_age = ('age','mean'),
    mean_score = ('score','mean'),
    purchase_rate = ('purchases','mean')))

         row_count   mean_age  mean_score  purchase_rate
city                                                    
Colombo        205  42.160976    0.612149       5.429268
Galle           84  39.809524    0.620845       5.285714
Jaffna          56  41.178571    0.591250       5.553571
Kandy           99  41.888889    0.582667       5.393939
Negombo         33  44.454545    0.586924       5.969697


In [890]:
# Age, income, score, and purchases can be used as numerical features.
# City can be encoded and used as a categorical feature.
# purchased and high_value can serve as potential prediction targets.
# Customer behavior differs across cities, especially in purchase rates.